In [1]:

import src.utils.notebook_ploting as nb_plot
import src.utils.features as features
from src.utils.notebook_setup import *

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

setup_pandas()

In [2]:
df = load_dataset()
df = df.sample(100000, random_state=42)
# essa amostra leva em torno de 25 s

In [3]:
numerical_features, categorical_features = features.split_features(df)

In [4]:
# separar features de label. Verificar o que fazer com as categóricas
    # transformar categóricas em flags binárias (provavelmente vai aumentar bastante o número de colunas)
features_df = df[numerical_features].copy()
target = df["label"]


In [5]:
# encoding do target (categorias → números)
# label_encoder possui o mapeamento e pode ser reutilizado
label_encoder = LabelEncoder()
target_encoded = label_encoder.fit_transform(target)


In [6]:
# tratamento de valores inválidos (provavelmente será necessário se houver algum tratamento de dado que gere um inf ou nan)
#features_df.isna().any().any()
#np.isinf(features_df).any().any()
#features_df = features_df.replace([np.inf, -np.inf], np.nan)
#features_df = features_df.fillna(features_df.median())

In [7]:
# cálculo da mutual information
    # mede o quão uma feature ajuda a prever o label
    # a partir de qual valor descartar essas features? (aparentemente apenas treinando um modelo para saber)
        # lembrete: treinar modelo com as melhores features top 10, 15, 20 ... verificar métrica (verificar com michael)
# random state funciona como seed para garantir reprodutibilidade do experimento
#
#array com scores
mi_scores = mutual_info_classif(features_df, target_encoded, random_state=1)
# mapeamento das features com mi_score
mi_scores_by_feature = pd.Series(mi_scores, index=features_df.columns).sort_values(ascending=False)

In [8]:
mi_scores_by_feature

bidirectional_bytes            0.10
bidirectional_mean_ps          0.10
bidirectional_max_ps           0.10
bidirectional_stddev_ps        0.09
dst2src_mean_ps                0.09
dst2src_bytes                  0.09
dst2src_max_ps                 0.09
dst2src_stddev_ps              0.08
src2dst_max_ps                 0.08
src2dst_mean_ps                0.07
src2dst_bytes                  0.07
src2dst_stddev_ps              0.07
src2dst_max_piat_ms            0.05
bidirectional_max_piat_ms      0.05
dst2src_max_piat_ms            0.05
bidirectional_mean_piat_ms     0.05
bidirectional_stddev_piat_ms   0.05
bidirectional_ack_packets      0.05
src2dst_stddev_piat_ms         0.05
dst2src_stddev_piat_ms         0.05
src2dst_mean_piat_ms           0.05
dst2src_mean_piat_ms           0.05
dst2src_duration_ms            0.05
src2dst_duration_ms            0.05
src2dst_rst_packets            0.04
bidirectional_packets          0.04
dst2src_ack_packets            0.04
dst2src_packets             

In [11]:
# dataframe final
# transforma o mapeamento em  tabela
mi_dataframe = mi_scores_by_feature.reset_index()
#display(mi_dataframe)
mi_dataframe.columns = ["feature", "mutual_information"]

In [12]:

# filtro por mi
#mi_dataframe = mi_dataframe[mi_dataframe["mutual_information"] > 0.02]
# filtro por top features
#mi_dataframe = mi_dataframe.head(15)

nb_plot.mi_bar_plot(mi_dataframe)